<a href="https://colab.research.google.com/github/dineshck2023-cpu/cv_workshop/blob/main/day02cv_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

YOLO OUTOUT

Class label → What the object is (e.g., person, car, dog)

Bounding box → Rectangle showing where the object is in the image

Confidence score → How sure the model is about the prediction

In [27]:
# Core libraries
import tensorflow as tf            # Main deep learning library
import numpy as np                # For numerical operations (arrays, math)
import matplotlib.pyplot as plt   # For plotting graphs and images

# Dataset
from tensorflow.keras.datasets import cifar10   # CIFAR-10 image dataset

# Model and layers
from tensorflow.keras import layers, models     # To build neural network models

# Pretrained model
from tensorflow.keras.applications import MobileNetV2   # Pretrained CNN model

# Utilities
from sklearn.metrics import classification_report, confusion_matrix  # Evaluation metrics
import seaborn as sns   # For better visualization (like heatmaps)

load dataset

In [28]:
# Load CIFAR-10 dataset (automatically split into train and test)
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# x_train → training images
# y_train → labels for training images
# x_test → testing images
# y_test → labels for testing images

# Print shapes to understand the data structure
print("Train shape:", x_train.shape)
# Example: (50000, 32, 32, 3) -> 50k images, each 32x32 with 3 color channels

print("Test shape:", x_test.shape)
# Example: (10000, 32, 32, 3)

Train shape: (50000, 32, 32, 3)
Test shape: (10000, 32, 32, 3)


Normalize data

In [29]:
# Normalize pixel values (0–255 -> 0–1)
x_train = x_train / 255.0
x_test = x_test / 255.0

Data augmentation

In [30]:
# Data augmentation helps the model learn better by showing slightly modified images
# This reduces overfitting and improves performance on new data

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),     # Randomly flips images left ↔ right
    layers.RandomRotation(0.1),          # Slightly rotates images (10% range)
    layers.RandomZoom(0.1)               # Slightly zooms in/out
])

load pretrained model

In [31]:
# Load MobileNetV2 model without the final classification layer (top layers)
base_model = MobileNetV2(input_shape=(32,32,3),
                         include_top=False,       # Remove default output layer
                         weights='imagenet')      # Use pretrained weights from ImageNet

# Freeze the base model so its learned features are not changed during training
base_model.trainable = False

/tmp/ipykernel_983/3232108708.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(input_shape=(32,32,3),


build advanced model

In [32]:


# Build Advanced Model
# Build custom model on top of the pretrained base model
model = models.Sequential([
    data_augmentation,              # Apply random transformations to input images
    base_model,                     # Use pretrained MobileNetV2 as feature extractor
    layers.GlobalAveragePooling2D(),# Convert feature maps into a single vector
    layers.BatchNormalization(),    # Normalize features to make training stable
    layers.Dense(128, activation="relu"),  # Fully connected layer for learning patterns
    layers.Dropout(0.5),            # Randomly drop neurons to reduce overfitting
    layers.Dense(10, activation="softmax") # Output layer for 10 classes
])

# Display model architecture
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_3 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 1, 1, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

compile model

In [33]:
# Compile the model (configure how it learns)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),  # Optimizer to update weights
    loss='sparse_categorical_crossentropy',  # Loss function for multi-class classification
    metrics=['accuracy']  # Metric to track performance
)

add callbacks

In [34]:
# Early stopping stops training when model stops improving (prevents overfitting)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  # Watch validation loss
    patience=3,  # Wait for 3 epochs before stopping if no improvement
    restore_best_weights=True  # Restore the best model weights after stopping
)

early stopping

In [35]:
# Early stopping stops training when model stops improving (prevents overfitting)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',          # Watch validation loss
    patience=3,                  # Wait for 3 epochs before stopping if no improvement
    restore_best_weights=True    # Restore the best model weights after stopping
)

train model

In [36]:
# Train the model on training data
history = model.fit(
    x_train, y_train,                 # Training data
    epochs=10,                        # Number of times the model sees the data
    validation_data=(x_test, y_test), # Check performance on test data after each epoch
    callbacks=[early_stop]            # Apply early stopping to avoid overfitting
)

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 96s 55ms/step - accuracy: 0.2381 - loss: 2.0678 - val_accuracy: 0.2759 - val_loss: 2.0869
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 84s 54ms/step - accuracy: 0.2614 - loss: 2.0152 - val_accuracy: 0.2867 - val_loss: 2.0677
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 86s 55ms/step - accuracy: 0.2634 - loss: 2.0071 - val_accuracy: 0.2945 - val_loss: 2.0617
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 88s 56ms/step - accuracy: 0.2660 - loss: 2.0027 - val_accuracy: 0.3066 - val_loss: 2.0468
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 85s 55ms/step - accuracy: 0.2687 - loss: 1.9991 - val_accuracy: 0.3109 - val_loss: 2.0590
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 85s 54ms/step - accuracy: 0.2682 - loss: 1.9954 - val_accuracy: 0.3124 - val_loss: 2.0858
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 88s 56ms/step - accuracy: 0.2695 - loss: 1.9883 - val_accuracy: 0.3065 - val_loss: 2.0622


In [ ]:
# Train the model on training data
history = model.fit(
    x_train, y_train,                 # Training data
    epochs=10,                        # Number of times the model sees the data
    validation_data=(x_test, y_test), # Check performance on test data after each epoch
    callbacks=[early_stop]            # Apply early stopping to avoid overfitting
)

Epoch 1/10
  48/1563 ━━━━━━━━━━━━━━━━━━━━ 1:12 48ms/step - accuracy: 0.2666 - loss: 1.9677

1. Load pre-trained model
2. Freeze initial layers
3. Unfreeze top layers
4. Train partially

In [ ]:
# Unfreeze top layers of base model
base_model.trainable = True

# Freeze initial layers, train only deeper layers
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train again
model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

evaluation model

In [ ]:
# Predictions
y_pred = np.argmax(model.predict(x_test), axis=1)

# Classification report
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

yolo object detection

In [ ]:
# 🔴 YOLO Object Detection

# Install ultralytics
!pip install ultralytics

In [ ]:
# Load YOLO model
from ultralytics import YOLO

yolo_model = YOLO('yolov8n.pt')


In [ ]:
# Run detection on image
results = yolo_model('https://ultralytics.com/images/bus.jpg', save=True)

Count detected objects

In [ ]:
# Count detected objects
for p in results:
    print("Objects detected:", len(p.boxes))

 Upload custom image

In [ ]:
# Upload custom image
from google.colab import files
uploaded = files.upload()


In [26]:
# Upload Custom Image
from google.colab import files
uploaded = files.upload()

for img in uploaded.keys():
    results = yolo_model(img, save=True)

Saving grassy-landscape-with-tree-raincloud.jpg to grassy-landscape-with-tree-raincloud (2).jpg

image 1/1 /content/grassy-landscape-with-tree-raincloud (2).jpg: 608x640 1 kite, 228.1ms
Speed: 6.6ms preprocess, 228.1ms inference, 3.1ms postprocess per image at shape (1, 3, 608, 640)
Results saved to /content/runs/detect/predict
